# Watching Stocks Move Through the Risk-Return Plane

In this notebook, we create an animated return-versus-volatility scatter plot for stocks of interest. Volatility, portfolio construction, and risk management have often been studied through distributions, summary statistics, and formal models, but what about the direct relationship between returns and volatility?

This question naturally rings a bell with modern portfolio theory à la Markowitz, where risk and return are studied together. However, instead of jumping immediately into optimization, this notebook takes a more visual and exploratory route. By plotting rolling returns against rolling volatility, we can watch how stocks move through the return-volatility plane over time, and perhaps notice patterns such as clustering, regime changes, stress periods, or differences in behavior across assets.

In this mini-project, we look at the qualitative side of this idea. Users can choose a list of tickers, select either simple returns or log returns, set the period, choose the moving-window length, and generate an animated scatter plot of return-versus-volatility evolution. The animation can also be saved as an MP4 for presentation or further study.

This is a work in progress, and I warmly invite contributions toward the quantitative development of this research.

1. ## Stock Data Selection

This first widget allows us to choose the stocks and date range for the analysis. Users may select tickers from the curated list, manually enter additional tickers, choose the start and end dates, and then load the corresponding adjusted close price data from Yahoo Finance.

To select multiple tickers from the list, hold **Ctrl** on Windows/Linux or **Command** on Mac while clicking. Custom tickers may be entered in the text box using commas, semicolons, or line breaks. Once the data are loaded, the notebook computes and stores the adjusted close prices, simple returns, and log returns for use in the later plotting and animation sections.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ipywidgets as widgets

from datetime import datetime
from IPython.display import display, clear_output


# ============================================================
# Ticker universe
# ============================================================

nyse_tickers = [
    'JPM', 'BAC', 'WFC', 'MS', 'GS',
    'XOM', 'CVX',
    'JNJ', 'PFE',
    'PG',
    'KO', 'PEP', 'DIS', 'NKE', 'VZ',
    'GE', 'IBM', 'BA', 'CAT', 'HON',
    'BRK-A', 'BRK-B'
]

nasdaq_tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'NVDA',
    'META', 'TSLA', 'NFLX', 'ADBE', 'INTC',
    'CSCO', 'CMCSA', 'QCOM', 'AMD', 'INTU',
    'MDLZ', 'BKNG', 'AMGN', 'ISRG', 'SBUX'
]

default_ticker_universe = sorted(nyse_tickers + nasdaq_tickers)
default_selected_tickers = ('AAPL', 'MSFT', 'NVDA', 'JPM')
default_start_date = pd.to_datetime('2018-01-01')
default_end_date = pd.to_datetime(datetime.now().strftime('%Y-%m-%d'))


# ============================================================
# Widgets
# ============================================================

ticker_selector = widgets.SelectMultiple(
    options=default_ticker_universe,
    value=default_selected_tickers,
    description='Tickers:',
    rows=14,
    layout=widgets.Layout(width='300px')
)

custom_ticker_text = widgets.Textarea(
    value='',
    placeholder='Optional: enter custom tickers separated by commas, e.g. SPY, QQQ, TSM',
    description='Custom:',
    layout=widgets.Layout(width='500px', height='120px')
)

start_date_picker = widgets.DatePicker(
    description='Start Date:',
    value=default_start_date
)

end_date_picker = widgets.DatePicker(
    description='End Date:',
    value=default_end_date
)

load_button = widgets.Button(
    description='Load Data',
    button_style='success',
    tooltip='Download adjusted close prices and compute returns',
    icon='download'
)

reset_button = widgets.Button(
    description='Reset',
    button_style='warning',
    tooltip='Reset ticker selection, dates, and output',
    icon='refresh'
)

output = widgets.Output()


# ============================================================
# Helper functions
# ============================================================

def parse_custom_tickers(text):
    """
    Parses manually entered tickers.

    Accepts comma-separated, semicolon-separated, or newline-separated input.
    Converts tickers to uppercase and removes empty entries.
    """
    if not text.strip():
        return []

    raw_items = text.replace('\n', ',').replace(';', ',').split(',')
    tickers = [item.strip().upper() for item in raw_items if item.strip()]

    return tickers


def get_selected_tickers():
    """
    Combines tickers chosen from the dropdown menu with manually entered tickers.
    Removes duplicates while preserving order.
    """
    selected_from_dropdown = list(ticker_selector.value)
    selected_custom = parse_custom_tickers(custom_ticker_text.value)

    selected_tickers = selected_from_dropdown + selected_custom
    selected_tickers = list(dict.fromkeys(selected_tickers))

    return selected_tickers


def reset_panel(button):
    """
    Resets the widget panel and clears previously stored data objects.
    """

    global selected_tickers
    global raw_price_data
    global adj_close_data
    global simple_return_data
    global log_return_data
    global unavailable_tickers

    ticker_selector.value = default_selected_tickers
    custom_ticker_text.value = ''
    start_date_picker.value = default_start_date
    end_date_picker.value = default_end_date

    selected_tickers = []
    raw_price_data = None
    adj_close_data = None
    simple_return_data = None
    log_return_data = None
    unavailable_tickers = []

    with output:
        clear_output()
        print("Selection panel has been reset.")


def load_stock_data(button):
    """
    Downloads adjusted close prices for the selected tickers and computes
    simple returns and log returns.

    The following global objects are created for later cells:

    selected_tickers
    raw_price_data
    adj_close_data
    simple_return_data
    log_return_data
    unavailable_tickers
    """

    global selected_tickers
    global raw_price_data
    global adj_close_data
    global simple_return_data
    global log_return_data
    global unavailable_tickers

    with output:
        clear_output()

        selected_tickers = get_selected_tickers()

        if len(selected_tickers) == 0:
            print("Error: Please select or enter at least one ticker.")
            return

        start_date = start_date_picker.value
        end_date = end_date_picker.value

        if start_date is None or end_date is None:
            print("Error: Please choose both a start date and an end date.")
            return

        start_date = pd.to_datetime(start_date).strftime('%Y-%m-%d')
        end_date = pd.to_datetime(end_date).strftime('%Y-%m-%d')

        if start_date >= end_date:
            print("Error: The start date must be earlier than the end date.")
            return

        print(f"Selected tickers: {selected_tickers}")
        print(f"Downloading data from {start_date} to {end_date}...")

        try:
            raw_price_data = yf.download(
                selected_tickers,
                start=start_date,
                end=end_date,
                auto_adjust=False,
                progress=False,
                group_by='column'
            )

            if raw_price_data.empty:
                print("Error: No data was downloaded.")
                print("Please check whether the tickers exist and whether the date range is valid.")
                return

            if isinstance(raw_price_data.columns, pd.MultiIndex):
                if 'Adj Close' not in raw_price_data.columns.get_level_values(0):
                    print("Error: Adjusted close prices were not found in the downloaded data.")
                    return

                adj_close_data = raw_price_data['Adj Close'].copy()

            else:
                if 'Adj Close' not in raw_price_data.columns:
                    print("Error: Adjusted close prices were not found in the downloaded data.")
                    return

                adj_close_data = raw_price_data[['Adj Close']].copy()
                adj_close_data.columns = selected_tickers[:1]

            # Remove columns that contain no usable adjusted close data.
            adj_close_data = adj_close_data.dropna(axis=1, how='all')

            available_tickers = list(adj_close_data.columns)
            unavailable_tickers = [
                ticker for ticker in selected_tickers
                if ticker not in available_tickers
            ]

            if adj_close_data.empty:
                print("Error: Adjusted close data is empty after removing unavailable tickers.")
                print("Possible reasons:")
                print("- The ticker does not exist.")
                print("- The ticker has no data for the selected date range.")
                print("- The ticker may have been delisted or listed after the chosen start date.")
                return

            simple_return_data = adj_close_data.pct_change().dropna(how='all')
            log_return_data = np.log(adj_close_data / adj_close_data.shift(1)).dropna(how='all')

            if len(unavailable_tickers) == 0:
                print("Success: All requested data acquired.")
            else:
                print("Warning: Data was acquired, but some tickers were unavailable.")
                print(f"Unavailable tickers: {unavailable_tickers}")
                print("Possible reasons:")
                print("- The ticker does not exist.")
                print("- The ticker has no data for the selected date range.")
                print("- The ticker may have been listed after the selected start date.")
                print("- The ticker may have been delisted.")

            print()
            print(f"Available tickers: {available_tickers}")
            print()

            # print("Adjusted close prices:")
            # display(adj_close_data.head())

            # print("Simple returns:")
            # display(simple_return_data.head())

            # print("Log returns:")
            # display(log_return_data.head())

        except Exception as e:
            print("Error: Data loading failed.")
            print(f"Details: {e}")


load_button.on_click(load_stock_data)
reset_button.on_click(reset_panel)


# ============================================================
# Display widget panel
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            """
            <h3>Stock Data Selection Panel</h3>
            <p>
            Select stocks from the dropdown menu. Hold <b>Ctrl</b> on Windows/Linux
            or <b>Command</b> on Mac to select multiple tickers.
            You may also enter your own tickers in the custom field.
            </p>
            """
        ),
        widgets.HBox([ticker_selector, custom_ticker_text]),
        widgets.HBox([start_date_picker, end_date_picker]),
        widgets.HBox([load_button, reset_button]),
        output
    ])
)

2. ## Exploratory Price and Return Distributions

Before constructing the animated return-versus-volatility scatter plot, it is useful to first inspect the selected stocks individually. This widget provides a quick exploratory view of the adjusted close price paths, the empirical distribution of simple returns, and the empirical distribution of log returns.

For the return and log-return histograms, the plot also displays the sample mean and sample median of each selected stock’s return series. This gives a preliminary sense of the location, spread, symmetry, and overlap of the selected return distributions before we move on to the rolling return-volatility analysis.

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


# ============================================================
# Plot Panel Widgets
# ============================================================

plot_type_radio = widgets.RadioButtons(
    options=[
        ('Price Plot', 'price'),
        ('Return Histogram', 'return'),
        ('Log-Return Histogram', 'log_return')
    ],
    value='price',
    description='Plot:',
    layout=widgets.Layout(width='300px')
)

plot_button = widgets.Button(
    description='Show Plot',
    button_style='success',
    tooltip='Generate selected plot',
    icon='bar-chart'
)

reset_plot_button = widgets.Button(
    description='Reset Plot',
    button_style='warning',
    tooltip='Clear plot output',
    icon='refresh'
)

plot_output = widgets.Output()


# ============================================================
# Plotting Function
# ============================================================

def show_selected_plot(button):
    """
    Displays one of the following plots:

    1. Adjusted close price plot
    2. Simple return histogram with mean and median lines
    3. Log-return histogram with mean and median lines

    For histograms:
    - Bars are semi-transparent.
    - Solid vertical lines represent means.
    - Dashed vertical lines represent medians.
    - Mean and median lines use the same color as the corresponding histogram.
    - If more than 10 tickers are selected, the ticker legend is omitted.
    """

    with plot_output:
        clear_output()

        required_objects = [
            'adj_close_data',
            'simple_return_data',
            'log_return_data'
        ]

        missing_objects = [
            obj for obj in required_objects
            if obj not in globals() or globals()[obj] is None
        ]

        if missing_objects:
            print("Error: Required data objects are missing.")
            print("Please load the stock data first using the previous selection panel.")
            print(f"Missing objects: {missing_objects}")
            return

        if adj_close_data.empty:
            print("Error: Adjusted close data is empty.")
            return

        plot_type = plot_type_radio.value
        tickers_to_plot = list(adj_close_data.columns)
        number_of_tickers = len(tickers_to_plot)

        fig, ax = plt.subplots(figsize=(12, 6))

        color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

        if plot_type == 'price':

            for i, ticker in enumerate(tickers_to_plot):
                color = color_cycle[i % len(color_cycle)]
                ax.plot(
                    adj_close_data.index,
                    adj_close_data[ticker],
                    label=ticker,
                    color=color
                )

            ax.set_title('Adjusted Close Price Plot')
            ax.set_xlabel('Date')
            ax.set_ylabel('Adjusted Close Price')

            if number_of_tickers <= 10:
                ax.legend(loc='upper right')
            else:
                ax.text(
                    0.01, 0.98,
                    "Legend cannot be displayed because there are too many tickers.",
                    transform=ax.transAxes,
                    ha='left',
                    va='top'
                )

        elif plot_type in ['return', 'log_return']:

            if plot_type == 'return':
                data_to_plot = simple_return_data
                title = 'Simple Return Histogram'
                x_label = 'Simple Return'
            else:
                data_to_plot = log_return_data
                title = 'Log-Return Histogram'
                x_label = 'Log Return'

            for i, ticker in enumerate(tickers_to_plot):
                if ticker not in data_to_plot.columns:
                    continue

                series = data_to_plot[ticker].dropna()

                if series.empty:
                    continue

                color = color_cycle[i % len(color_cycle)]

                ax.hist(
                    series,
                    bins=50,
                    alpha=0.45,
                    label=ticker,
                    color=color
                )

                mean_value = series.mean()
                median_value = series.median()

                ax.axvline(
                    mean_value,
                    color=color,
                    linestyle='-',
                    linewidth=2
                )

                ax.axvline(
                    median_value,
                    color=color,
                    linestyle='--',
                    linewidth=2
                )

            ax.set_title(title)
            ax.set_xlabel(x_label)
            ax.set_ylabel('Frequency')

            ax.text(
                0.01, 0.98,
                "Colorized solid vertical lines are means.\nColorized dashed vertical lines are medians.",
                transform=ax.transAxes,
                ha='left',
                va='top'
            )

            if number_of_tickers <= 10:
                ax.legend(loc='upper right')
            else:
                ax.text(
                    0.99, 0.98,
                    "Legend cannot be displayed\nbecause there are too many tickers.",
                    transform=ax.transAxes,
                    ha='right',
                    va='top'
                )

        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


def reset_plot_panel(button):
    """
    Clears the plot output panel.
    """

    with plot_output:
        clear_output()
        print("Plot panel has been reset.")


plot_button.on_click(show_selected_plot)
reset_plot_button.on_click(reset_plot_panel)


# ============================================================
# Display Plot Panel
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            """
            <h3>Exploratory Plot Panel</h3>
            <p>
            Use this panel to display the adjusted close price plot,
            the simple return histogram, or the log-return histogram.
            For histograms, solid vertical lines represent means and dashed
            vertical lines represent medians.
            </p>
            """
        ),
        plot_type_radio,
        widgets.HBox([plot_button, reset_plot_button]),
        plot_output
    ])
)

3. ## Drill-In Return Distribution

After viewing the broader price and return plots, this widget allows us to zoom in on the return distribution of at most two selected stocks. The user may choose one or two tickers from the loaded data and then plot either the simple-return histogram or the log-return histogram.

For each selected ticker, the histogram includes solid vertical lines for the sample mean and the mean plus or minus one, two, and three standard deviations. It also includes dashed vertical lines for the median, first quartile, and third quartile. This provides a more detailed view of the distribution’s center, spread, skewness, and tail behavior for the chosen stock or pair of stocks.

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


# ============================================================
# Drill-In Histogram Panel Widgets
# ============================================================

def get_available_ticker_options():
    """
    Retrieves available tickers from adj_close_data.
    """
    if 'adj_close_data' not in globals() or adj_close_data is None or adj_close_data.empty:
        return ['None']

    return ['None'] + list(adj_close_data.columns)


ticker_1_dropdown = widgets.Dropdown(
    options=get_available_ticker_options(),
    value='None',
    description='Ticker 1:',
    layout=widgets.Layout(width='300px')
)

ticker_2_dropdown = widgets.Dropdown(
    options=get_available_ticker_options(),
    value='None',
    description='Ticker 2:',
    layout=widgets.Layout(width='300px')
)

return_button = widgets.Button(
    description='Plot Return',
    button_style='success',
    tooltip='Plot simple return histogram',
    icon='bar-chart'
)

log_return_button = widgets.Button(
    description='Plot Log-Return',
    button_style='info',
    tooltip='Plot log-return histogram',
    icon='bar-chart'
)

reset_drill_button = widgets.Button(
    description='Reset',
    button_style='warning',
    tooltip='Reset ticker choices and clear output',
    icon='refresh'
)

drill_output = widgets.Output()


# ============================================================
# Helper Functions
# ============================================================

def refresh_drill_dropdown_options():
    """
    Refreshes dropdown options based on currently loaded adjusted close data.
    This is useful if the user reloads data from the first widget.
    """
    options = get_available_ticker_options()

    ticker_1_dropdown.options = options
    ticker_2_dropdown.options = options

    if ticker_1_dropdown.value not in options:
        ticker_1_dropdown.value = 'None'

    if ticker_2_dropdown.value not in options:
        ticker_2_dropdown.value = 'None'


def get_drill_tickers():
    """
    Gets one or two selected tickers, removing 'None' and duplicates.
    """
    selected = [
        ticker_1_dropdown.value,
        ticker_2_dropdown.value
    ]

    selected = [ticker for ticker in selected if ticker != 'None']
    selected = list(dict.fromkeys(selected))

    return selected


def plot_drill_histogram(return_type):
    """
    Plots a histogram for one or two selected tickers.

    return_type may be:
    - 'return'
    - 'log_return'

    Solid vertical lines:
    - mean
    - mean +/- 1 standard deviation
    - mean +/- 2 standard deviations
    - mean +/- 3 standard deviations

    Dashed vertical lines:
    - median
    - Q1
    - Q3
    """

    with drill_output:
        clear_output()

        refresh_drill_dropdown_options()

        if return_type == 'return':
            if 'simple_return_data' not in globals() or simple_return_data is None or simple_return_data.empty:
                print("Error: Simple return data is unavailable. Please load the stock data first.")
                return

            data_to_plot = simple_return_data
            title = 'Drill-In Simple Return Histogram'
            x_label = 'Simple Return'

        elif return_type == 'log_return':
            if 'log_return_data' not in globals() or log_return_data is None or log_return_data.empty:
                print("Error: Log-return data is unavailable. Please load the stock data first.")
                return

            data_to_plot = log_return_data
            title = 'Drill-In Log-Return Histogram'
            x_label = 'Log Return'

        else:
            print("Error: Unknown return type.")
            return

        selected_drill_tickers = get_drill_tickers()

        if len(selected_drill_tickers) == 0:
            print("Please select at least one ticker to drill into.")
            return

        if len(selected_drill_tickers) > 2:
            print("Please select at most two tickers.")
            return

        fig, ax = plt.subplots(figsize=(12, 6))

        color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

        for i, ticker in enumerate(selected_drill_tickers):
            if ticker not in data_to_plot.columns:
                print(f"Warning: {ticker} is not available in the selected return data.")
                continue

            series = data_to_plot[ticker].dropna()

            if series.empty:
                print(f"Warning: {ticker} has no usable return data.")
                continue

            color = color_cycle[i % len(color_cycle)]

            ax.hist(
                series,
                bins=60,
                alpha=0.45,
                label=ticker,
                color=color
            )

            mean_value = series.mean()
            std_value = series.std()
            median_value = series.median()
            q1_value = series.quantile(0.25)
            q3_value = series.quantile(0.75)

            # Mean and standard deviation lines: solid
            ax.axvline(mean_value, color=color, linestyle='-', linewidth=2)

            for k in [1, 2, 3]:
                ax.axvline(mean_value + k * std_value, color=color, linestyle='-', linewidth=1.5)
                ax.axvline(mean_value - k * std_value, color=color, linestyle='-', linewidth=1.5)

            # Median and quartile lines: dashed
            ax.axvline(median_value, color=color, linestyle='--', linewidth=2)
            ax.axvline(q1_value, color=color, linestyle='--', linewidth=1.5)
            ax.axvline(q3_value, color=color, linestyle='--', linewidth=1.5)

        ax.set_title(title)
        ax.set_xlabel(x_label)
        ax.set_ylabel('Frequency')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')

        plt.tight_layout()
        plt.show()


def plot_return_histogram(button):
    plot_drill_histogram(return_type='return')


def plot_log_return_histogram(button):
    plot_drill_histogram(return_type='log_return')


def reset_drill_panel(button):
    """
    Resets the drill-in panel.
    """
    refresh_drill_dropdown_options()

    ticker_1_dropdown.value = 'None'
    ticker_2_dropdown.value = 'None'

    with drill_output:
        clear_output()
        print("Drill-in histogram panel has been reset.")


return_button.on_click(plot_return_histogram)
log_return_button.on_click(plot_log_return_histogram)
reset_drill_button.on_click(reset_drill_panel)


# ============================================================
# Display Drill-In Histogram Panel
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            """
            <h3>Drill-In Return Distribution Panel</h3>
            """
        ),
        widgets.HBox([ticker_1_dropdown, ticker_2_dropdown]),
        widgets.HBox([return_button, log_return_button, reset_drill_button]),
        drill_output
    ])
)

4. ## Volatility and IQR Comparison

This widget compares the dispersion of the selected stocks’ return distributions using two simple summary measures: standard deviation and interquartile range. The standard deviation is shown as a bar plot, while the interquartile range is shown as a line plot on a secondary vertical axis.

The user may choose whether the comparison is based on simple returns or log returns, and may sort the tickers in increasing or decreasing order by either standard deviation or interquartile range. This gives a quick cross-sectional view of which selected stocks exhibit higher overall volatility and which have wider central return dispersion.

In [17]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


# ============================================================
# Volatility and IQR Comparison Panel Widgets
# ============================================================

order_metric_radio = widgets.RadioButtons(
    options=[
        ('Order by Standard Deviation', 'std'),
        ('Order by IQR', 'iqr')
    ],
    value='std',
    description='Metric:',
    layout=widgets.Layout(width='320px')
)

order_direction_radio = widgets.RadioButtons(
    options=[
        ('Increasing', 'increasing'),
        ('Decreasing', 'decreasing')
    ],
    value='decreasing',
    description='Order:',
    layout=widgets.Layout(width='250px')
)

return_type_radio_iqr = widgets.RadioButtons(
    options=[
        ('Simple Return', 'return'),
        ('Log Return', 'log_return')
    ],
    value='return',
    description='Data:',
    layout=widgets.Layout(width='250px')
)

plot_iqr_button = widgets.Button(
    description='Show Plot',
    button_style='success',
    tooltip='Plot standard deviation and IQR comparison',
    icon='bar-chart'
)

reset_iqr_button = widgets.Button(
    description='Reset',
    button_style='warning',
    tooltip='Reset choices and clear output',
    icon='refresh'
)

iqr_output = widgets.Output()


# ============================================================
# Plotting Function
# ============================================================

def plot_std_iqr_comparison(button):
    """
    Plots standard deviation as bars and IQR as a line on a secondary y-axis.

    The plot is based on all tickers selected in the first data-selection panel.

    Users may choose:
    - simple returns or log returns,
    - ordering by standard deviation or IQR,
    - increasing or decreasing order.
    """

    with iqr_output:
        clear_output()

        if return_type_radio_iqr.value == 'return':
            if 'simple_return_data' not in globals() or simple_return_data is None or simple_return_data.empty:
                print("Error: Simple return data is unavailable. Please load the stock data first.")
                return

            data_to_plot = simple_return_data
            title_return_type = 'Simple Returns'

        else:
            if 'log_return_data' not in globals() or log_return_data is None or log_return_data.empty:
                print("Error: Log-return data is unavailable. Please load the stock data first.")
                return

            data_to_plot = log_return_data
            title_return_type = 'Log Returns'

        stats_data = []

        for ticker in data_to_plot.columns:
            series = data_to_plot[ticker].dropna()

            if series.empty:
                continue

            std_value = series.std()
            q1_value = series.quantile(0.25)
            q3_value = series.quantile(0.75)
            iqr_value = q3_value - q1_value

            stats_data.append({
                'Ticker': ticker,
                'Standard Deviation': std_value,
                'IQR': iqr_value
            })

        if len(stats_data) == 0:
            print("Error: No usable return data was found.")
            return

        stats_df = pd.DataFrame(stats_data)

        sort_column = 'Standard Deviation' if order_metric_radio.value == 'std' else 'IQR'
        ascending = True if order_direction_radio.value == 'increasing' else False

        stats_df = stats_df.sort_values(by=sort_column, ascending=ascending).reset_index(drop=True)

        fig, ax1 = plt.subplots(figsize=(13, 6))

        x_positions = range(len(stats_df))

        bars = ax1.bar(
            x_positions,
            stats_df['Standard Deviation'],
            alpha=0.75,
            label='Standard Deviation'
        )

        ax1.set_xlabel('Ticker')
        ax1.set_ylabel('Standard Deviation')
        ax1.set_xticks(x_positions)
        ax1.set_xticklabels(stats_df['Ticker'], rotation=45, ha='right')
        ax1.grid(True, axis='y', alpha=0.3)

        ax2 = ax1.twinx()

        line = ax2.plot(
            x_positions,
            stats_df['IQR'],
            marker='o',
            linewidth=2,
            label='IQR'
        )

        ax2.set_ylabel('Interquartile Range')

        ax1.set_title(
            f'Standard Deviation and IQR Comparison Based on {title_return_type}'
        )

        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()

        ax1.legend(
            lines_1 + lines_2,
            labels_1 + labels_2,
            loc='upper right'
        )

        plt.tight_layout()
        plt.show()

        print("Summary statistics:")
        display(stats_df)


def reset_iqr_panel(button):
    """
    Resets the volatility-IQR comparison panel.
    """

    order_metric_radio.value = 'std'
    order_direction_radio.value = 'decreasing'
    return_type_radio_iqr.value = 'return'

    with iqr_output:
        clear_output()
        print("Volatility and IQR comparison panel has been reset.")


plot_iqr_button.on_click(plot_std_iqr_comparison)
reset_iqr_button.on_click(reset_iqr_panel)


# ============================================================
# Display Volatility and IQR Comparison Panel
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            """
            <h3>Volatility and IQR Comparison Panel</h3>
            """
        ),
        widgets.HBox([return_type_radio_iqr, order_metric_radio, order_direction_radio]),
        widgets.HBox([plot_iqr_button, reset_iqr_button]),
        iqr_output
    ])
)

5. ## Mean and Median Comparison

This widget compares the average location of each selected stock’s return distribution using the sample mean and sample median. The mean is shown as a bar plot, while the median is shown as a line plot over the same ticker ordering.

The user may choose whether the comparison is based on simple returns or log returns, and may order the tickers in increasing or decreasing order by either standard deviation or interquartile range. This helps reveal whether the central tendency of each return distribution changes meaningfully when viewed through the mean versus the median, especially when the distributions are skewed or affected by extreme observations.

In [19]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


# ============================================================
# Mean and Median Comparison Panel Widgets
# ============================================================

mean_median_return_type_radio = widgets.RadioButtons(
    options=[
        ('Simple Return', 'return'),
        ('Log Return', 'log_return')
    ],
    value='return',
    description='Data:',
    layout=widgets.Layout(width='250px')
)

mean_median_order_metric_radio = widgets.RadioButtons(
    options=[
        ('Order by Standard Deviation', 'std'),
        ('Order by IQR', 'iqr')
    ],
    value='std',
    description='Metric:',
    layout=widgets.Layout(width='320px')
)

mean_median_order_direction_radio = widgets.RadioButtons(
    options=[
        ('Increasing', 'increasing'),
        ('Decreasing', 'decreasing')
    ],
    value='decreasing',
    description='Order:',
    layout=widgets.Layout(width='250px')
)

plot_mean_median_button = widgets.Button(
    description='Show Plot',
    button_style='success',
    tooltip='Plot mean and median comparison',
    icon='bar-chart'
)

reset_mean_median_button = widgets.Button(
    description='Reset',
    button_style='warning',
    tooltip='Reset choices and clear output',
    icon='refresh'
)

mean_median_output = widgets.Output()


# ============================================================
# Plotting Function
# ============================================================

def plot_mean_median_comparison(button):
    """
    Plots mean as bars and median as a line.

    The plot is based on all tickers selected in the first data-selection panel.

    Users may choose:
    - simple returns or log returns,
    - ordering by standard deviation or IQR,
    - increasing or decreasing order.
    """

    with mean_median_output:
        clear_output()

        if mean_median_return_type_radio.value == 'return':
            if 'simple_return_data' not in globals() or simple_return_data is None or simple_return_data.empty:
                print("Error: Simple return data is unavailable. Please load the stock data first.")
                return

            data_to_plot = simple_return_data
            title_return_type = 'Simple Returns'

        else:
            if 'log_return_data' not in globals() or log_return_data is None or log_return_data.empty:
                print("Error: Log-return data is unavailable. Please load the stock data first.")
                return

            data_to_plot = log_return_data
            title_return_type = 'Log Returns'

        stats_data = []

        for ticker in data_to_plot.columns:
            series = data_to_plot[ticker].dropna()

            if series.empty:
                continue

            mean_value = series.mean()
            median_value = series.median()
            std_value = series.std()
            q1_value = series.quantile(0.25)
            q3_value = series.quantile(0.75)
            iqr_value = q3_value - q1_value

            stats_data.append({
                'Ticker': ticker,
                'Mean': mean_value,
                'Median': median_value,
                'Standard Deviation': std_value,
                'IQR': iqr_value
            })

        if len(stats_data) == 0:
            print("Error: No usable return data was found.")
            return

        stats_df = pd.DataFrame(stats_data)

        sort_column = 'Standard Deviation' if mean_median_order_metric_radio.value == 'std' else 'IQR'
        ascending = True if mean_median_order_direction_radio.value == 'increasing' else False

        stats_df = stats_df.sort_values(by=sort_column, ascending=ascending).reset_index(drop=True)

        fig, ax = plt.subplots(figsize=(13, 6))

        x_positions = range(len(stats_df))

        ax.bar(
            x_positions,
            stats_df['Mean'],
            alpha=0.75,
            label='Mean'
        )

        ax.plot(
            x_positions,
            stats_df['Median'],
            marker='o',
            linewidth=2,
            label='Median'
        )

        ax.axhline(
            0,
            linewidth=1,
            linestyle='--',
            alpha=0.7
        )

        ax.set_title(
            f'Mean and Median Comparison Based on {title_return_type}'
        )
        ax.set_xlabel('Ticker')
        ax.set_ylabel('Return')
        ax.set_xticks(x_positions)
        ax.set_xticklabels(stats_df['Ticker'], rotation=45, ha='right')
        ax.grid(True, axis='y', alpha=0.3)
        ax.legend(loc='upper right')

        plt.tight_layout()
        plt.show()

        print("Summary statistics:")
        display(stats_df)


def reset_mean_median_panel(button):
    """
    Resets the mean-median comparison panel.
    """

    mean_median_return_type_radio.value = 'return'
    mean_median_order_metric_radio.value = 'std'
    mean_median_order_direction_radio.value = 'decreasing'

    with mean_median_output:
        clear_output()
        print("Mean and median comparison panel has been reset.")


plot_mean_median_button.on_click(plot_mean_median_comparison)
reset_mean_median_button.on_click(reset_mean_median_panel)


# ============================================================
# Display Mean and Median Comparison Panel
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            """
            <h3>Mean and Median Comparison Panel</h3>
            """
        ),
        widgets.HBox([
            mean_median_return_type_radio,
            mean_median_order_metric_radio,
            mean_median_order_direction_radio
        ]),
        widgets.HBox([plot_mean_median_button, reset_mean_median_button]),
        mean_median_output
    ])
)

6. ## Animated Return-Volatility Scatter Plot

This widget generates the main animated scatter plot for the notebook. It allows us to choose a shorter date range from the loaded data, select either simple returns or log returns, and animate either rolling mean versus rolling standard deviation or rolling median versus rolling interquartile range.

The rolling window determines how many trading days are used to compute each point in the scatter plot. The step determines how many trading days the rolling window moves forward between animation frames; a smaller step gives a smoother animation, while a larger step makes the animation shorter and faster to generate. The interval determines the playback delay between frames, measured in milliseconds.

Each ticker is shown as a moving dot, with its ticker symbol annotated in bold. The path traced out by each dot shows how that stock’s return-volatility profile evolves through time.

The animation may become too large to display directly in the notebook, especially when many tickers, long date ranges, or small step sizes are used. To reduce display issues, the notebook increases Matplotlib’s animation embed limit before generating or re-running the animation.

**Note:** *Give the program time to generate the animation. It may take a while.*

In [ ]:
import matplotlib.pyplot as plt

# Increase the animation embed limit to allow larger animations (e.g., 50 MB)
plt.rcParams['animation.embed_limit'] = 90.0

print(f"Matplotlib animation embed limit set to: {plt.rcParams['animation.embed_limit']} MB")
print("You may re-run the animation cells above if you wish to generate them with the increased limit.")

Matplotlib animation embed limit set to: 90.0 MB
You may re-run the animation cells above if you wish to generate them with the increased limit.


In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import ipywidgets as widgets

from datetime import datetime
from IPython.display import display, clear_output, HTML


# ============================================================
# Animated Return-Volatility Panel Widgets
# ============================================================

def get_loaded_date_bounds():
    """
    Gets the available date range from the loaded adjusted close data.
    """
    if 'adj_close_data' not in globals() or adj_close_data is None or adj_close_data.empty:
        today = pd.to_datetime(datetime.now().strftime('%Y-%m-%d'))
        return today, today

    return adj_close_data.index.min(), adj_close_data.index.max()


loaded_start_date, loaded_end_date = get_loaded_date_bounds()

animation_start_date_picker = widgets.DatePicker(
    description='Start Date:',
    value=loaded_start_date.to_pydatetime()
)

animation_end_date_picker = widgets.DatePicker(
    description='End Date:',
    value=loaded_end_date.to_pydatetime()
)

animation_return_type_radio = widgets.RadioButtons(
    options=[
        ('Simple Return', 'return'),
        ('Log Return', 'log_return')
    ],
    value='return',
    description='Data:',
    layout=widgets.Layout(width='250px')
)

animation_metric_radio = widgets.RadioButtons(
    options=[
        ('Mean vs Standard Deviation', 'mean_std'),
        ('Median vs IQR', 'median_iqr')
    ],
    value='mean_std',
    description='Animation:',
    layout=widgets.Layout(width='340px')
)

rolling_window_slider = widgets.IntSlider(
    value=60,
    min=5,
    max=252,
    step=1,
    description='Window:',
    continuous_update=False,
    layout=widgets.Layout(width='420px')
)

animation_step_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=30,
    step=1,
    description='Step:',
    continuous_update=False,
    layout=widgets.Layout(width='420px')
)

animation_interval_slider = widgets.IntSlider(
    value=250,
    min=50,
    max=1000,
    step=50,
    description='Interval:',
    continuous_update=False,
    layout=widgets.Layout(width='420px')
)

show_animation_button = widgets.Button(
    description='Show Animation',
    button_style='success',
    tooltip='Generate animated scatter plot',
    icon='play'
)

reset_animation_button = widgets.Button(
    description='Reset',
    button_style='warning',
    tooltip='Reset choices and clear output',
    icon='refresh'
)

animation_output = widgets.Output()


# ============================================================
# Helper Functions
# ============================================================

def refresh_animation_date_bounds():
    """
    Refreshes date picker bounds and values from the currently loaded data.
    """
    loaded_start_date, loaded_end_date = get_loaded_date_bounds()

    animation_start_date_picker.value = loaded_start_date.to_pydatetime()
    animation_end_date_picker.value = loaded_end_date.to_pydatetime()


def compute_rolling_animation_stats(data, start_date, end_date, window, step, metric_type):
    """
    Computes rolling statistics for the animation.

    metric_type:
    - 'mean_std': x = rolling mean, y = rolling standard deviation
    - 'median_iqr': x = rolling median, y = rolling IQR
    """

    data_period = data.loc[start_date:end_date].copy()
    data_period = data_period.dropna(axis=1, how='all')

    if data_period.empty:
        return None

    if len(data_period) < window:
        return None

    frame_dates = []
    frame_stats = []

    for end_idx in range(window, len(data_period) + 1, step):
        window_data = data_period.iloc[end_idx - window:end_idx]
        current_date = data_period.index[end_idx - 1]

        if metric_type == 'mean_std':
            x_values = window_data.mean()
            y_values = window_data.std()
        else:
            x_values = window_data.median()
            q1_values = window_data.quantile(0.25)
            q3_values = window_data.quantile(0.75)
            y_values = q3_values - q1_values

        stats = pd.DataFrame({
            'x': x_values,
            'y': y_values
        }).dropna()

        if not stats.empty:
            frame_dates.append(current_date)
            frame_stats.append(stats)

    if len(frame_stats) == 0:
        return None

    return frame_dates, frame_stats


def build_animation(button):
    """
    Builds an animated scatter plot of either:

    1. rolling mean vs rolling standard deviation, or
    2. rolling median vs rolling IQR.

    Dots are annotated with ticker labels in bold.
    Loci and current dots use different color cycles.

    The date corresponding to each frame is displayed at the upper-left
    corner of the plot.
    """

    global return_volatility_animation

    with animation_output:
        clear_output()
        print("Working... Please be patient.")

        if animation_return_type_radio.value == 'return':
            if 'simple_return_data' not in globals() or simple_return_data is None or simple_return_data.empty:
                clear_output()
                print("ERROR: Simple return data is unavailable. Please load stock data first.")
                return

            data_to_animate = simple_return_data.copy()
            return_type_label = 'Simple Return'

        else:
            if 'log_return_data' not in globals() or log_return_data is None or log_return_data.empty:
                clear_output()
                print("ERROR: Log-return data is unavailable. Please load stock data first.")
                return

            data_to_animate = log_return_data.copy()
            return_type_label = 'Log Return'

        start_date = animation_start_date_picker.value
        end_date = animation_end_date_picker.value

        if start_date is None or end_date is None:
            clear_output()
            print("ERROR: Please choose both a start date and an end date.")
            return

        start_date = pd.to_datetime(start_date)
        end_date = pd.to_datetime(end_date)

        if start_date >= end_date:
            clear_output()
            print(f"ERROR: Date range is invalid. Start date {start_date.date()} must be earlier than end date {end_date.date()}.")
            return

        available_start = data_to_animate.index.min()
        available_end = data_to_animate.index.max()

        if start_date < available_start or end_date > available_end:
            clear_output()
            print("ERROR: The chosen animation period is outside the loaded data range.")
            print(f"Chosen range: {start_date.date()} to {end_date.date()}")
            print(f"Available range: {available_start.date()} to {available_end.date()}")
            return

        window = rolling_window_slider.value
        step = animation_step_slider.value
        interval = animation_interval_slider.value
        metric_type = animation_metric_radio.value

        result = compute_rolling_animation_stats(
            data=data_to_animate,
            start_date=start_date,
            end_date=end_date,
            window=window,
            step=step,
            metric_type=metric_type
        )

        if result is None:
            clear_output()
            print("ERROR: Not enough usable data for the selected period and rolling-window length.")
            print(f"Chosen range: {start_date.date()} to {end_date.date()}")
            print(f"Rolling window length: {window}")
            return

        frame_dates, frame_stats = result

        all_tickers = sorted(set().union(*[set(frame.index) for frame in frame_stats]))

        if len(all_tickers) == 0:
            clear_output()
            print("ERROR: No usable ticker data found for the selected animation period.")
            return

        if metric_type == 'mean_std':
            x_label = f'Rolling Mean of {return_type_label}'
            y_label = f'Rolling Standard Deviation of {return_type_label}'
            title_label = 'Rolling Mean vs Rolling Standard Deviation'
        else:
            x_label = f'Rolling Median of {return_type_label}'
            y_label = f'Rolling IQR of {return_type_label}'
            title_label = 'Rolling Median vs Rolling IQR'

        x_all = pd.concat([frame['x'] for frame in frame_stats])
        y_all = pd.concat([frame['y'] for frame in frame_stats])

        x_padding = 0.10 * (x_all.max() - x_all.min()) if x_all.max() != x_all.min() else 0.001
        y_padding = 0.10 * (y_all.max() - y_all.min()) if y_all.max() != y_all.min() else 0.001

        x_min = x_all.min() - x_padding
        x_max = x_all.max() + x_padding
        y_min = max(0, y_all.min() - y_padding)
        y_max = y_all.max() + y_padding

        fig, ax = plt.subplots(figsize=(12, 7))

        locus_colors = plt.cm.tab20(np.linspace(0, 1, max(len(all_tickers), 1)))
        dot_colors = plt.cm.Set1(np.linspace(0, 1, max(len(all_tickers), 1)))

        ticker_to_locus_color = {
            ticker: locus_colors[i % len(locus_colors)]
            for i, ticker in enumerate(all_tickers)
        }

        ticker_to_dot_color = {
            ticker: dot_colors[i % len(dot_colors)]
            for i, ticker in enumerate(all_tickers)
        }

        ticker_paths = {
            ticker: {'x': [], 'y': []}
            for ticker in all_tickers
        }

        def init():
            ax.clear()
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            ax.set_xlabel(x_label)
            ax.set_ylabel(y_label)
            ax.grid(True, alpha=0.3)
            return []

        def update(frame_index):
            ax.clear()

            current_stats = frame_stats[frame_index]
            current_date = frame_dates[frame_index]

            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            ax.set_xlabel(x_label)
            ax.set_ylabel(y_label)
            ax.grid(True, alpha=0.3)

            ax.set_title(
                f'{title_label}\n'
                f'{return_type_label}; Rolling Window = {window}'
            )

            # Date label on the upper-left of each frame
            ax.text(
                0.02,
                0.96,
                f'Date: {current_date.date()}',
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=12,
                fontweight='bold',
                bbox=dict(
                    boxstyle='round,pad=0.3',
                    facecolor='white',
                    edgecolor='black',
                    alpha=0.75
                )
            )

            for ticker in all_tickers:
                if ticker in current_stats.index:
                    x_value = current_stats.loc[ticker, 'x']
                    y_value = current_stats.loc[ticker, 'y']

                    ticker_paths[ticker]['x'].append(x_value)
                    ticker_paths[ticker]['y'].append(y_value)

                path_x = ticker_paths[ticker]['x']
                path_y = ticker_paths[ticker]['y']

                if len(path_x) > 0:
                    ax.plot(
                        path_x,
                        path_y,
                        color=ticker_to_locus_color[ticker],
                        linewidth=1.5,
                        alpha=0.75
                    )

                    ax.scatter(
                        path_x[-1],
                        path_y[-1],
                        color=ticker_to_dot_color[ticker],
                        s=90,
                        edgecolor='black',
                        linewidth=0.8,
                        zorder=3
                    )

                    ax.annotate(
                        ticker,
                        xy=(path_x[-1], path_y[-1]),
                        xytext=(6, 6),
                        textcoords='offset points',
                        fontsize=9,
                        fontweight='bold'
                    )

            return []

        return_volatility_animation = animation.FuncAnimation(
            fig,
            update,
            frames=len(frame_stats),
            init_func=init,
            interval=interval,
            blit=False,
            repeat=False
        )

        plt.close(fig)

        clear_output()
        display(HTML(return_volatility_animation.to_jshtml()))

        print("Animation generated successfully.")
        print("The animation object is stored as `return_volatility_animation`.")


def reset_animation_panel(button):
    """
    Resets the animation panel.
    """

    refresh_animation_date_bounds()

    animation_return_type_radio.value = 'return'
    animation_metric_radio.value = 'mean_std'
    rolling_window_slider.value = 60
    animation_step_slider.value = 5
    animation_interval_slider.value = 250

    with animation_output:
        clear_output()
        print("Animation panel has been reset.")


show_animation_button.on_click(build_animation)
reset_animation_button.on_click(reset_animation_panel)


# ============================================================
# Display Animated Return-Volatility Panel
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            """
            <h3>Animated Return-Volatility Scatter Plot Panel</h3>
            """
        ),
        widgets.HBox([
            animation_start_date_picker,
            animation_end_date_picker
        ]),
        widgets.HBox([
            animation_return_type_radio,
            animation_metric_radio
        ]),
        rolling_window_slider,
        animation_step_slider,
        animation_interval_slider,
        widgets.HBox([
            show_animation_button,
            reset_animation_button
        ]),
        animation_output
    ])
)

7. ## Save Animation as MP4

This final widget allows the generated animation to be saved as an MP4 file. Users may enter a custom filename before saving, or leave the filename field blank and let the notebook generate a default filename based on the animation settings.

The save function uses the animation object created in the previous section, so the animation must be generated before it can be exported. If saving fails, the most common reason is that FFmpeg is unavailable or not properly configured in the notebook environment.

In [21]:
import os
import re
import ipywidgets as widgets

from IPython.display import display, clear_output


# ============================================================
# MP4 Save Panel Widgets
# ============================================================

video_filename_text = widgets.Text(
    value='',
    placeholder='Optional: enter filename, e.g. my_return_volatility_animation',
    description='Filename:',
    layout=widgets.Layout(width='520px')
)

save_mp4_button = widgets.Button(
    description='Save MP4',
    button_style='success',
    tooltip='Save the generated animation as an MP4 file',
    icon='save'
)

reset_save_button = widgets.Button(
    description='Reset',
    button_style='warning',
    tooltip='Clear filename and output',
    icon='refresh'
)

save_output = widgets.Output()


# ============================================================
# Helper Functions
# ============================================================

def clean_filename(filename):
    """
    Cleans a user-entered filename so that it is safer to use as a file name.
    """
    filename = filename.strip()

    if filename == '':
        return ''

    filename = re.sub(r'[\\/*?:"<>|]', '_', filename)

    if not filename.lower().endswith('.mp4'):
        filename += '.mp4'

    return filename


def get_default_animation_filename():
    """
    Creates a default MP4 filename based on the animation choices.
    """

    if 'animation_return_type_radio' in globals():
        return_type = animation_return_type_radio.value
    else:
        return_type = 'return'

    if 'animation_metric_radio' in globals():
        metric_type = animation_metric_radio.value
    else:
        metric_type = 'mean_std'

    if 'rolling_window_slider' in globals():
        window = rolling_window_slider.value
    else:
        window = 'window'

    if return_type == 'return':
        return_label = 'simple_return'
    else:
        return_label = 'log_return'

    if metric_type == 'mean_std':
        metric_label = 'mean_vs_std'
    else:
        metric_label = 'median_vs_iqr'

    default_filename = f'{return_label}_{metric_label}_window_{window}.mp4'

    return default_filename


def save_animation_as_mp4(button):
    """
    Saves the generated animation as an MP4 file.

    If the user enters a filename, that filename is used.
    Otherwise, a default filename is generated from the animation settings.
    """

    with save_output:
        clear_output()

        if 'return_volatility_animation' not in globals() or return_volatility_animation is None:
            print("Error: No animation object was found.")
            print("Please generate the animation first before saving it as an MP4.")
            return

        user_filename = clean_filename(video_filename_text.value)

        if user_filename == '':
            output_filename = get_default_animation_filename()
        else:
            output_filename = user_filename

        try:
            print(f"Saving animation as: {output_filename}")
            print("This may take a little while, especially for long animations.")

            return_volatility_animation.save(
                output_filename,
                writer='ffmpeg',
                fps=10,
                dpi=150
            )

            if os.path.exists(output_filename):
                print("Success: Animation saved as MP4.")
                print(f"Saved file: {output_filename}")
            else:
                print("Warning: The save command finished, but the file was not found.")

        except Exception as e:
            print("Error: Failed to save animation as MP4.")
            print("Possible reasons:")
            print("- FFmpeg is not available.")
            print("- The animation object was not generated correctly.")
            print("- The filename or folder location is invalid.")
            print()
            print(f"Details: {e}")


def reset_save_panel(button):
    """
    Resets the MP4 save panel.
    """

    video_filename_text.value = ''

    with save_output:
        clear_output()
        print("MP4 save panel has been reset.")


save_mp4_button.on_click(save_animation_as_mp4)
reset_save_button.on_click(reset_save_panel)


# ============================================================
# Display MP4 Save Panel
# ============================================================

display(
    widgets.VBox([
        widgets.HTML(
            """
            <h3>Save Animation as MP4</h3>
            """
        ),
        video_filename_text,
        widgets.HBox([
            save_mp4_button,
            reset_save_button
        ]),
        save_output
    ])
)

## Summary

This notebook develops an interactive visual workflow for studying the relationship between stock returns and volatility. Starting from a selected set of tickers, the notebook downloads adjusted close price data, computes simple returns and log returns, and then provides several exploratory widgets for inspecting price paths, return distributions, summary statistics, and rolling return-volatility behavior.

The first widget allows users to choose stocks from a curated ticker list, add custom tickers, select a date range, and generate the adjusted close, return, and log-return datasets. The next widgets provide static exploratory plots: price paths, overlapping return histograms, drill-in histograms for one or two tickers, standard deviation versus IQR comparisons, and mean versus median comparisons. These views give a preliminary sense of how the selected assets differ in location, dispersion, skewness, and tail behavior.

The main object of the notebook is the animated return-volatility scatter plot. Users can choose a shorter time period, select simple returns or log returns, and animate either rolling mean versus rolling standard deviation or rolling median versus rolling IQR. Each ticker moves through the scatter plot over time, tracing a locus that represents the evolution of its empirical return-volatility profile.

At this stage, the notebook is primarily qualitative and exploratory. It is designed to help us see patterns, clusters, regime changes, and unusual behavior before imposing a formal model. However, the broader goal is to move toward a quantitative theory of these trajectories: to ask whether the shapes, directions, speeds, clustering behavior, or transitions of these return-volatility loci can be measured, compared, classified, and eventually used in portfolio construction, risk management, or market-regime analysis.

In that sense, this notebook is both a visualization tool and a starting point for research. The visual plots help generate hypotheses; the next step is to develop quantitative methods that can test, formalize, and possibly operationalize those hypotheses.